In [40]:
import sys
sys.path.append('..')
from encoder import FMEncoder, compute_spectrogram_mel, compute_spectrogram_cqt
import fm_ddsp
import torch

%load_ext autoreload
%autoreload 2

from fm_ddsp import fm_renderer, make_mod_matrix
from loss import multiscale_stft_loss
import torch
import numpy as np
import matplotlib.pyplot as plt

f0 = 440.0
ratios = torch.tensor([3.0, 1.0, 1.0, 1.0])
levels = torch.tensor([0.8, 1.0, 0.0, 0.0])

mod_matrix = make_mod_matrix(torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]))
carrier_weights = torch.tensor([0.0, 1.0, 0.0, 0.0])
audio = fm_renderer(f0, ratios, levels, mod_matrix, carrier_weights, 16000, 1.0)
print("Audio shape:", audio.shape)
print("Import successful")





ImportError: cannot import name 'compute_spectrogram_mel' from 'encoder' (C:\Users\Marcus\Documents\FM_DDSP\python\notebooks\..\encoder.py)

In [42]:
Fs = 16000
n_fft = 4096
n_mels = 256


audio = fm_ddsp.sin_wav(440, Fs, 1.0)
mel_spec = compute_spectrogram(audio, Fs, n_fft, n_mels)
print("mel spec shape:", mel_spec.shape)

plt.figure(figsize=(10, 4))
plt.plot(mel_spec.squeeze().numpy())
plt.colorbar()
plt.xlabel('Time Frames')
plt.ylabel('Mel bins')
plt.title('Mel Spectrogram')
plt.show()

NameError: name 'compute_spectrogram' is not defined

In [43]:
from nnAudio.features import CQT2010v2
Fs = 16000
hop_size = 512
bins_per_octave = 24
n_octaves = 7
f0 = 440/2

audio = fm_ddsp.sin_wav(f0, Fs, 1.0)
audio = audio + fm_ddsp.sin_wav(f0*3/2, Fs, 1.0)
audio = audio + fm_ddsp.sin_wav(f0*2, Fs, 1.0)
audio = audio + fm_ddsp.sin_wav(f0//2, Fs, 1.0)
spec = compute_spectrogram_cqt(audio)
# cqt_transform = CQT2010v2(sr = Fs,
#                               hop_length = hop_size,
#                               n_bins = bins_per_octave*n_octaves,
#                               bins_per_octave = bins_per_octave)
# spec = cqt_transform(audio.unsqueeze(0))
# spec = spec.abs()
# spec = torch.log1p(spec)
# spec = spec.mean(dim=2)
print(spec.shape)
print("cqt spec shape:", spec.shape)

plt.figure(figsize=(10, 4))
plt.plot(spec)
print(f"plot shape: {spec.shape}")
plt.ylabel('Amplitude')
plt.xlabel('CQT bins')
plt.title('CQT Spectrogram')
plt.show()

NameError: name 'compute_spectrogram_cqt' is not defined

In [ ]:
f0 = 440.0
ratios = torch.tensor([3.0, 1.0, 1.0, 1.0])
levels = torch.tensor([0.8, 1.0, 0.0, 0.0])
mod_matrix = make_mod_matrix(torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]))
carrier_weights = torch.tensor([0.0, 1.0, 0.0, 0.0])
fm_audio = fm_renderer(f0, ratios, levels, mod_matrix, carrier_weights, 16000, 1.0)

mel_spec = mel_transform(fm_audio)
plt.figure(figsize=(10, 4))
plt.imshow(torch.log1p(mel_spec).numpy(),
           aspect='auto',
           origin='lower')
plt.colorbar()
plt.xlabel('Time frames')
plt.ylabel('Mel bins')
plt.title('FM Mel Spectrogram')
plt.show()

In [ ]:
from dataset import FMDataset

dataset = FMDataset(n_examples = 100)
print(f"Dataset size: {len(dataset)}")

example = dataset[0]
for key, val, in example.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}, {val.shape}")
    else:
        print(f"{key}, {val}")

In [ ]:
from encoder import FMEncoder
from dataset import FMDataset

datset = FMDataset(n_examples = 100)
encoder = FMEncoder()

example = dataset[0]
spec = example['spectrogram'].unsqueeze(0)
params = encoder(spec)
for key, val in params.items():
    print(f"{key}: {val.shape}")

In [44]:
import torch, json
import os
print(os.listdir('../data/test_run'))
# load one spectrogram
spec = torch.load('../data/test_run/spec_01.pt', weights_only = False)
print(spec.shape)

# load matching params
with open('../data/test_run/params_01.json') as f:
    params = json.load(f)
print(params)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../data/test_run'

In [ ]:
!python generate_dataset.py --n_examples 10000 --save_dir ../data/test_run --overwrite True

In [67]:
dataset = FMDataset("C:\\Users\\Marcus\\Documents\\FM_DDSP\\python\\data\\test_run")
print(len(dataset))
print(dataset[10])

10000
({'f0': 415.3046975799451, 'algorithm': 'algo_4', 'mod_values': tensor([0.1565, 0.2536, 0.0000, 0.4469, 0.0000, 0.0000, 0.1388]), 'ratios': tensor([6., 2., 4., 5.]), 'levels': tensor([0.7642, 0.2947, 0.3440, 0.2010]), 'carrier_weights': tensor([0.0000, 0.0000, 0.4332, 0.6155])}, array([3.1127763e-04, 3.2387735e-04, 2.9949180e-04, 2.9592947e-04,
       3.1494137e-04, 2.9980947e-04, 3.1703213e-04, 3.1400161e-04,
       3.1737587e-04, 3.1024113e-04, 3.1025545e-04, 2.9108388e-04,
       3.1320465e-04, 3.0929319e-04, 3.0060511e-04, 2.8156713e-04,
       3.0560928e-04, 3.0406681e-04, 2.8852539e-04, 2.8755888e-04,
       2.8385274e-04, 2.8690137e-04, 2.8937802e-04, 2.8222107e-04,
       2.7857543e-04, 2.7767572e-04, 2.8196524e-04, 2.8251359e-04,
       2.8829917e-04, 2.8597054e-04, 2.9085678e-04, 2.7385959e-04,
       2.7931866e-04, 2.8052166e-04, 2.8485022e-04, 2.7362758e-04,
       2.8196839e-04, 2.8389192e-04, 2.7570347e-04, 2.7613365e-04,
       2.6702322e-04, 2.5693656e-04, 2.69610